══════════════════════════════════════════════════════════════
 WEEK 10  |  DAY 2  |  PROMPT ENGINEERING

══════════════════════════════════════════════════════════════

 LEARNING OBJECTIVES
 ───────────────────
 1. Write clear, effective prompts that get reliable results
 2. Use few-shot prompting (give examples in the prompt)
 3. Ask for structured output (JSON, CSV, tables)
 4. Chain prompts — use one LLM response as input to the next
 5. Use Pydantic to enforce a schema on LLM JSON output
 6. Choose the right model tier and estimate API call costs

 TIME:  ~55 minutes

 YOUTUBE
 ───────
 Search: "prompt engineering tutorial Python LLM"
 Search: "few shot prompting explained examples"
 Search: "ChatGPT prompt engineering best practices"
 Search: "Pydantic structured output OpenAI Python"
 Search: "OpenAI API pricing tokens explained"

 INSTALL:
   pip install pydantic openai

 NOTE: You need an API key from Day 1 to run the live examples.
       If you don't have one yet, read the concepts and do Exercise 1
       (which doesn't require an API call).

 ─────────────────────────────────────────────────────────────
 ARCHITECTURE DECISION
 ─────────────────────
 Choosing between: zero-shot vs few-shot examples vs chain-of-thought vs fine-tuning.
 Rule of thumb: start zero-shot. Add few-shot when output format is critical. Use chain-of-thought for multi-step reasoning. Fine-tune only when prompting cannot close the gap after several weeks of iteration.

══════════════════════════════════════════════════════════════

In [ ]:
import os
import json

══════════════════════════════════════════════════════════════
 CONCEPT 1 — WHAT MAKES A GOOD PROMPT?

══════════════════════════════════════════════════════════════

A good prompt is:
  1. SPECIFIC about the task
  2. SPECIFIC about the output format
  3. SPECIFIC about what to do when unsure

BAD prompt:
  "Analyze this data."

GOOD prompt:
  "You are a data analyst. Given the following CSV data, identify the top 3
   products by revenue. Respond ONLY with a JSON array like:
   [{'product': '...', 'revenue': ...}]
   If data is missing or unclear, return an empty array []."

THE THREE ROLES:
  system   — persona and rules for the model ("You are a...")
  user     — the actual request
  assistant— what the model responds (used in multi-turn conversations)

EXAMPLE ──────────────────────────────────────────────────────

In [ ]:
print("=" * 55)
print("CONCEPT 1: Good vs Bad Prompts")
print("=" * 55)
print()
print("BAD:")
bad_prompt = "Tell me about the data."
print(f"  User: {bad_prompt}")
print("  Problem: What data? What format? What level of detail?")
print()
print("GOOD:")
good_prompt = """
You are a data analyst. Given this sales summary:
  Product A: revenue=50000, units=200
  Product B: revenue=30000, units=350
  Product C: revenue=80000, units=100

List the products ranked by revenue (highest first).
Respond ONLY with a JSON array:
[{"rank": 1, "product": "...", "revenue": ...}, ...]
"""
print(f"  User: {good_prompt.strip()}")
print("  Better: specific role, specific data, specific format")

══════════════════════════════════════════════════════════════
 EXERCISE 1

══════════════════════════════════════════════════════════════
Rewrite each bad prompt as a good one. Write your answers as comments.

Bad prompt 1:
  "Summarize this text."
  Rewrite it for a data engineer who needs a 2-sentence summary of a technical doc.

Bad prompt 2:
  "Is this customer happy?"
  Rewrite it to classify sentiment as positive/negative/neutral and return JSON.

Bad prompt 3:
  "Fix my code."
  Rewrite it so the model explains WHAT was wrong before giving the fix.

Expected output:
    3 well-written prompts with specific roles, formats, and edge-case handling

══════════════════════════════════════════════════════════════
 CONCEPT 2 — FEW-SHOT PROMPTING

══════════════════════════════════════════════════════════════

Few-shot = give the model 2-3 examples of the input/output you want.
The model learns the pattern from your examples and applies it to new input.
Very useful when you need a specific format or style.

EXAMPLE ──────────────────────────────────────────────────────

In [ ]:
print()
print("=" * 55)
print("CONCEPT 2: Few-Shot Prompt Template")
print("=" * 55)

few_shot_prompt = """
You classify customer support tickets into categories.

Examples:
  Input:  "My payment was declined twice."
  Output: billing

  Input:  "The app crashes when I open the settings menu."
  Output: technical

  Input:  "I never received my order from last week."
  Output: shipping

Now classify this ticket:
  Input:  "I was charged twice for the same subscription."
  Output:
"""

print(few_shot_prompt)
print("Expected answer: billing")
print()
print("The model learns the pattern: input → single lowercase category word")

# To run with an API (uncomment once you have a key):
# from openai import OpenAI
# client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
# response = client.chat.completions.create(
#     model="gpt-4o-mini",
#     messages=[{"role": "user", "content": few_shot_prompt}],
#     max_tokens=10
# )
# print("API answer:", response.choices[0].message.content.strip())

══════════════════════════════════════════════════════════════
 EXERCISE 2

══════════════════════════════════════════════════════════════
Build a few-shot prompt that extracts structured data from messy text.
Provide 2 examples, then test with a new input.

Task: extract "city" and "population" from a sentence.

Example inputs/outputs:
  Input:  "Tel Aviv has a population of about 460,000 people."
  Output: {"city": "Tel Aviv", "population": 460000}

  Input:  "Jerusalem, with 970,000 residents, is the largest city."
  Output: {"city": "Jerusalem", "population": 970000}

New input to classify:
  "Haifa is home to around 285,000 inhabitants."

Write the full few-shot prompt string (no API call needed — just the prompt).
As a comment, write what you expect the model to output.

Expected output:
    A prompt string with 2 examples and 1 new input
    Comment: {"city": "Haifa", "population": 285000}

══════════════════════════════════════════════════════════════
 CONCEPT 3 — PROMPT CHAINING

══════════════════════════════════════════════════════════════

Chaining = use the output of one LLM call as the input to the next.
Good for complex tasks that are too hard to do in one step.

Example pipeline:
  Step 1: Extract raw facts from a long document  → bullet points
  Step 2: Turn bullet points into a SQL INSERT statement
  Step 3: Validate the SQL for correctness

EXAMPLE ──────────────────────────────────────────────────────

In [ ]:
print()
print("=" * 55)
print("CONCEPT 3: Prompt Chaining Template")
print("=" * 55)

def call_llm(prompt, system="You are a helpful assistant.", model="gpt-4o-mini"):
    """Template function — replace with real API call."""
    # from openai import OpenAI
    # client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
    # response = client.chat.completions.create(
    #     model=model,
    #     messages=[
    #         {"role": "system", "content": system},
    #         {"role": "user",   "content": prompt}
    #     ],
    #     max_tokens=300
    # )
    # return response.choices[0].message.content.strip()
    return "[API response would appear here]"   # placeholder

# Chain example:
raw_text = """
  Meeting notes - Q3 Review - July 2025
  Sales: 2.4M, up 12% vs Q2. Top product: Laptop Pro (35% of revenue).
  Concerns: logistics delays causing 8% return rate.
  Next steps: hire 2 logistics managers, launch Laptop Pro 2 in September.
"""

# Step 1: Extract key facts
step1_prompt  = f"Extract 4 key facts from these meeting notes as bullet points:\n{raw_text}"
facts         = call_llm(step1_prompt, system="You are a concise note-taker.")
print("Step 1 output (key facts):")
print(facts)

# Step 2: Turn facts into an action plan
step2_prompt  = f"Turn these facts into a prioritized action plan with 3 items:\n{facts}"
action_plan   = call_llm(step2_prompt, system="You are a project manager.")
print("\nStep 2 output (action plan):")
print(action_plan)

══════════════════════════════════════════════════════════════
 EXERCISE 3

══════════════════════════════════════════════════════════════
Build a prompt chain that does the following:
  Step 1: Given a raw CSV row as text, extract the fields as JSON
  Step 2: Given the JSON, write a Python print statement that displays the data nicely

Raw input:
  "Alice Cohen,30,Data Engineer,Tel Aviv,75000"

Expected Step 1 output (JSON):
  {"name": "Alice Cohen", "age": 30, "role": "Data Engineer",
   "city": "Tel Aviv", "salary": 75000}

Expected Step 2 output (Python code):
  print(f"Name: Alice Cohen | Role: Data Engineer | City: Tel Aviv | Salary: 75,000")

Write the two prompts as strings (call_llm() will return a placeholder for now).
When you have an API key, replace the call_llm() function body with a real call.

In [ ]:
raw_csv_row = "Alice Cohen,30,Data Engineer,Tel Aviv,75000"

══════════════════════════════════════════════════════════════
 CONCEPT 4 — STRUCTURED OUTPUTS (PYDANTIC)
══════════════════════════════════════════════════════════════

Asking for JSON in a prompt works sometimes — but it breaks in production.
The LLM might add text around the JSON, rename fields, or return invalid syntax.

Structured outputs solve this: you define a Pydantic schema and the LLM
is forced to match it exactly. No extra text, no missing fields.

WHAT IS PYDANTIC?
  Pydantic is a Python library for data validation.
  You define a class that inherits from BaseModel.
  Each field has a name and a type — Pydantic enforces both.

  class EmployeeRecord(BaseModel):
      name:   str
      age:    int
      salary: int
      active: bool

  If the LLM returns {"name": "Dan", "age": "thirty"} — Pydantic raises an error.
  If the LLM returns {"name": "Dan", "age": 30, "salary": 80000, "active": true} — it works.

TWO WAYS TO USE STRUCTURED OUTPUTS:
  1. OpenAI response_format — pass your Pydantic model directly to the API
     completion = client.beta.chat.completions.parse(response_format=MyModel)
     result = completion.choices[0].message.parsed   # already a MyModel instance

  2. Manual — ask for JSON in the prompt, parse with json.loads(), validate with Pydantic

WHY THIS MATTERS IN PRODUCTION:
  - Your code can use result.field_name with autocomplete (no dict key guessing)
  - Missing or wrong fields are caught immediately, not 3 steps later
  - The data contract between your prompt and your code is explicit

INSTALL:
  pip install pydantic   (usually already installed)

EXAMPLE ──────────────────────────────────────────────────────

In [ ]:

print()
print("=" * 55)
print("CONCEPT 4: Structured Outputs with Pydantic")
print("=" * 55)

from pydantic import BaseModel
import json

# Define the expected schema
class ProductReview(BaseModel):
    product:   str
    sentiment: str    # "positive", "negative", or "neutral"
    score:     int    # 1 to 5
    reason:    str

# Parse and validate a JSON string (simulates structured LLM output)
raw_json = '{"product": "Laptop Pro", "sentiment": "positive", "score": 4, "reason": "Fast and reliable"}'
review = ProductReview(**json.loads(raw_json))

print(f"Product:   {review.product}")
print(f"Sentiment: {review.sentiment}")
print(f"Score:     {review.score}/5")
print(f"Reason:    {review.reason}")
print()

# Pydantic also catches bad data:
try:
    bad = ProductReview(**{"product": "Keyboard", "sentiment": "positive", "score": "not a number", "reason": "ok"})
except Exception as e:
    print(f"Validation error caught: score must be an integer")

print()
print("With OpenAI structured output (uncomment once you have an API key):")
print("  client.beta.chat.completions.parse(model=..., response_format=ProductReview)")
print("  result = completion.choices[0].message.parsed")
print("  print(result.product, result.score)   # IDE autocomplete works on result")

# With the OpenAI API (uncomment once you have a key):
# from openai import OpenAI
# client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
# completion = client.beta.chat.completions.parse(
#     model="gpt-4o-mini",
#     messages=[
#         {"role": "system", "content": "Extract review data as structured JSON."},
#         {"role": "user",   "content": "The Laptop Pro is fantastic — fast, reliable. I'd give it 4 stars."}
#     ],
#     response_format=ProductReview
# )
# review = completion.choices[0].message.parsed
# print(review.product, review.score, review.sentiment)


══════════════════════════════════════════════════════════════
 EXERCISE 4
══════════════════════════════════════════════════════════════
Define a Pydantic model called SalesRecord with these fields:
  employee_name:  str
  city:           str
  sales_amount:   int    (in NIS)
  quarter:        str    (e.g. "Q3")
  above_target:   bool   (True if sales_amount > 50,000)

Parse valid_json using your model and print each field with a label.

Then try to parse invalid_json (missing the "quarter" field).
Wrap it in try/except and print a helpful error message if validation fails.

Expected output:
    Employee: Dana Levi
    City:     Tel Aviv
    Sales:    67,000 NIS
    Quarter:  Q3
    Target:   Yes
    ---
    Validation error: field "quarter" is missing

In [ ]:

import json
from pydantic import BaseModel

valid_json   = '{"employee_name": "Dana Levi", "city": "Tel Aviv", "sales_amount": 67000, "quarter": "Q3", "above_target": true}'
invalid_json = '{"employee_name": "Ron Ben", "city": "Haifa", "sales_amount": 43000, "above_target": false}'






══════════════════════════════════════════════════════════════
 CONCEPT 5 — COST & MODEL SELECTION
══════════════════════════════════════════════════════════════

Not all tasks need the most powerful model.
Using the wrong tier is the most common way to waste money on LLM APIs.

MODEL TIERS (approximate 2025 pricing per 1 million tokens):

  CHEAP   gpt-4o-mini / claude-haiku
          Input: ~$0.15   Output: ~$0.60
          Use for: classification, extraction, short Q&A, high-volume tasks
          Response quality: good for structured tasks, weaker on open-ended reasoning

  MID     gpt-4o / claude-sonnet
          Input: ~$3.00   Output: ~$15.00
          Use for: complex analysis, code generation, quality writing
          Response quality: strong across most tasks

  EXPENSIVE  o1 / claude-opus
          Input: ~$15.00  Output: ~$75.00
          Use for: novel math/logic, hard multi-step reasoning, research
          Response quality: best available — only use when mid is not good enough

DECISION RULE:
  1. Start with the cheap model
  2. If output quality is unacceptable, move to mid
  3. Only use expensive when mid fails on a task that truly requires deep reasoning

TOKEN COUNTING (approximate):
  1 token ≈ 0.75 English words
  100 words ≈ 133 tokens
  A short email (~150 words) ≈ 200 tokens
  A 1-page document (~500 words) ≈ 667 tokens

COST FORMULA:
  cost = (input_tokens × input_price + output_tokens × output_price) / 1,000,000

PROMPT COMPRESSION TIPS (reduce input tokens = reduce cost):
  - Use bullet points in system prompts, not full paragraphs
  - Remove few-shot examples once the pattern is learned
  - Truncate large documents — send only the relevant section
  - Cache responses for repeated identical inputs

EXAMPLE ──────────────────────────────────────────────────────

In [ ]:

print()
print("=" * 55)
print("CONCEPT 5: Cost & Model Selection")
print("=" * 55)

# Model pricing per 1 million tokens (approximate 2025 prices, check current rates):
pricing = {
    "cheap":     {"input": 0.15,  "output": 0.60,  "example": "gpt-4o-mini / claude-haiku"},
    "mid":       {"input": 3.00,  "output": 15.00, "example": "gpt-4o / claude-sonnet"},
    "expensive": {"input": 15.00, "output": 75.00, "example": "o1 / claude-opus"},
}

def estimate_cost(input_tokens, output_tokens, tier):
    """Estimate cost for one API call or a batch."""
    p = pricing[tier]
    cost = (input_tokens * p["input"] + output_tokens * p["output"]) / 1_000_000
    return cost

# Example: classify 1 customer review
input_tk  = 30    # system prompt (~15) + review text (~15)
output_tk = 3     # "positive" = 1 token

cost_cheap = estimate_cost(input_tk, output_tk, "cheap")
cost_mid   = estimate_cost(input_tk, output_tk, "mid")

print(f"Classifying 1 review:")
print(f"  cheap tier: ${cost_cheap:.6f}")
print(f"  mid tier:   ${cost_mid:.6f}")
print(f"  cheap is {cost_mid / cost_cheap:.0f}x cheaper for this task")
print()

# Scale to 10,000 reviews:
scale = 10_000
print(f"Classifying {scale:,} reviews:")
print(f"  cheap tier total: ${estimate_cost(input_tk * scale, output_tk * scale, 'cheap'):.4f}")
print(f"  mid tier total:   ${estimate_cost(input_tk * scale, output_tk * scale, 'mid'):.4f}")

print()
print("Model selection rule:")
print("  cheap:     classification, extraction, short answers, high volume")
print("  mid:       analysis, code, complex instructions, quality writing")
print("  expensive: novel reasoning, hard problems, multi-step logic")
print()
print("PROMPT COMPRESSION TIPS:")
print("  - Remove verbose examples once the model knows the pattern")
print("  - Use bullet points in system prompts, not full paragraphs")
print("  - Truncate large inputs — send only the relevant section")
print("  - Cache responses for repeated identical prompts")


══════════════════════════════════════════════════════════════
 EXERCISE 5
══════════════════════════════════════════════════════════════
For each task in the tasks list below, write 3 comments:
  1. Which model tier (cheap / mid / expensive) — and why
  2. Estimated token count (input + output) for one call
  3. Estimated total cost using model_pricing above

Use the estimate_cost(input_tokens, output_tokens, tier) function
you will write as part of this exercise.

Then answer: which task has the highest total cost, and why?

Expected output (approximate values):
    Task 1 (classify 10k reviews):
      Tier: cheap  — simple binary label, no reasoning needed
      Tokens: ~20 input × 10000 = 200,000 input, ~3 output × 10000 = 30,000 output
      Cost: ~$0.048

    Task 2 (write 2000-word report):
      Tier: mid  — complex writing, needs quality
      Tokens: ~50 input, ~2700 output
      Cost: ~$0.041

    Task 3 (extract 500 contacts):
      Tier: cheap  — structured extraction, no reasoning
      Tokens: ~30 × 500 = 15,000 input, ~20 × 500 = 10,000 output
      Cost: ~$0.008

    Task 4 (novel math problem):
      Tier: expensive  — requires careful multi-step reasoning
      Tokens: ~100 input, ~300 output
      Cost: ~$0.024

In [ ]:

model_pricing = {
    "cheap":     {"input": 0.15,  "output": 0.60,  "example": "gpt-4o-mini / claude-haiku"},
    "mid":       {"input": 3.00,  "output": 15.00, "example": "gpt-4o / claude-sonnet"},
    "expensive": {"input": 15.00, "output": 75.00, "example": "o1 / claude-opus"},
}

tasks = [
    "Classify 10,000 customer reviews (1 sentence each) as positive / negative / neutral",
    "Write a 2,000-word technical report on database design patterns",
    "Extract name, email, phone from 500 contact form submissions",
    "Solve a novel multi-step math problem requiring careful reasoning",
]






══════════════════════════════════════════════════════════════
 CONCEPT 6 — PROMPT VERSIONING: MANAGE PROMPTS LIKE CODE
══════════════════════════════════════════════════════════════
 Prompts are as important as code. A bad prompt change can break production
 just like a bad code deploy. Treat prompts with the same discipline:
   - Version them (v1, v2, v3...)
   - Test before promoting
   - Roll back if they regress

 PromptRegistry pattern:
   - Store prompts in a dict: {name → {version → {template, description, created}}}
   - Retrieve the active version by name
   - A/B test: run two versions on the same inputs, compare scores

 In production: PromptLayer, LangSmith, or LangFuse store prompts in a DB
 with versioning, tagging, and deployment status. Here we implement the
 core pattern in plain Python so you understand what those tools do.

 EXAMPLE ──────────────────────────────────────────────────────

In [ ]:
from datetime import datetime


class PromptRegistry:
    """Version-controlled prompt storage."""

    def __init__(self):
        self._registry: dict = {}   # {name: {version: {template, description, active}}}

    def register(self, name: str, version: str, template: str,
                 description: str = "") -> None:
        if name not in self._registry:
            self._registry[name] = {}
        self._registry[name][version] = {
            "template":    template,
            "description": description,
            "created":     datetime.now().strftime("%Y-%m-%d"),
            "active":      False,
        }

    def activate(self, name: str, version: str) -> None:
        """Set the active version for a prompt name."""
        for v in self._registry.get(name, {}):
            self._registry[name][v]["active"] = False
        self._registry[name][version]["active"] = True

    def get(self, name: str, version: str = None) -> str:
        """Return the template for name@version (or active version)."""
        versions = self._registry.get(name, {})
        if version:
            return versions[version]["template"]
        for v, meta in versions.items():
            if meta["active"]:
                return meta["template"]
        raise KeyError(f"No active version found for '{name}'")

    def format(self, name: str, version: str = None, **kwargs) -> str:
        return self.get(name, version).format(**kwargs)

    def list_versions(self, name: str) -> list:
        return [
            {"version": v, "description": m["description"], "active": m["active"]}
            for v, m in self._registry.get(name, {}).items()
        ]


# Build a registry with two versions of a customer support prompt
registry = PromptRegistry()

registry.register(
    "support_classifier", "v1",
    template="Classify this support ticket: {ticket}\nCategory:",
    description="Simple zero-shot classifier",
)
registry.register(
    "support_classifier", "v2",
    template=(
        "You are a support specialist. Classify the ticket below into ONE of:\n"
        "billing, technical, account, general\n\nTicket: {ticket}\nCategory:"
    ),
    description="Role-prompted with explicit categories",
)

registry.activate("support_classifier", "v2")

print("Versions:")
for v in registry.list_versions("support_classifier"):
    active = " [ACTIVE]" if v["active"] else ""
    print(f"  {v['version']}{active}: {v['description']}")

ticket = "I was charged twice this month"
print("\nv1 prompt:")
print(registry.format("support_classifier", "v1", ticket=ticket))
print("\nv2 prompt (active):")
print(registry.format("support_classifier", ticket=ticket))

══════════════════════════════════════════════════════════════
 EXERCISE 6 — PROMPT VERSIONING AND A/B COMPARISON
══════════════════════════════════════════════════════════════
 Build a PromptRegistry using the starting data below and add one method
 that does not appear in the example:

   compare_versions(self, name, v1, v2, inputs: list[dict], scorer: callable) -> dict
     Runs format(name, v1, **inp) and format(name, v2, **inp) for each input.
     Calls scorer(formatted_prompt) to get a score (float) for each.
     Returns:
       {"v1_avg": float, "v2_avg": float, "winner": "v1" or "v2"}

 Register both prompts below. Use the provided scorer function.
 Call compare_versions on all 3 inputs and print the result dict.

 Expected output:
     v1_avg: X.XX  v2_avg: X.XX  winner: vX

 --- starting data ---

In [ ]:
from datetime import datetime

# Scorer: awards 1 point per word in the formatted prompt (longer = more detailed)
# Replace with a real LLM judge in production
def word_count_scorer(prompt: str) -> float:
    return float(len(prompt.split()))

# Inputs to test both prompt versions against
test_inputs = [
    {"product": "DataPipeline Pro", "customer": "Alice", "issue": "slow exports"},
    {"product": "DataPipeline Pro", "customer": "Bob",   "issue": "login error"},
    {"product": "DataPipeline Pro", "customer": "Carol", "issue": "missing data"},
]

# Prompt templates to register:
PROMPT_V1 = "Help {customer} with their {product} issue: {issue}"
PROMPT_V2 = (
    "You are a senior support engineer for {product}.\n"
    "Customer name: {customer}\n"
    "Issue reported: {issue}\n"
    "Provide a clear, step-by-step resolution plan."
)

# ══════════════════════════════════════════════════════════════
#  CONCEPT 7 -- LLM AS DATA VALIDATOR
# ══════════════════════════════════════════════════════════════
#
#  Traditional data validation checks rules: 'age must be > 0'.
#  LLM validation adds semantic understanding: 'does this dataset
#  look like real customer data? Is N/A appearing where it should
#  not? Are these prices realistic?'
#
#  Pattern used in ETL pipelines (Stage 1 -- pre-ingestion check):
#    1. Take a sample of the raw data (20-50 rows as JSON)
#    2. Send it to the LLM with the expected schema description
#    3. Ask for a structured validation report:
#         anomalies     : list of specific issues found
#         quality_score : 0-10 (0 = unusable, 10 = perfect)
#         schema_match  : True/False
#
#  Pydantic enforces the LLM response structure.
#
#  EXAMPLE ----------------------------------------------------------

In [ ]:
import os
import json
from dotenv import load_dotenv
from pydantic import BaseModel

load_dotenv()
GROQ_API_KEY = os.environ.get('GROQ_API_KEY', '')

# ── Pydantic model for the validation report ──────────────────
class ValidationReport(BaseModel):
    schema_match:   bool
    anomalies:      list
    quality_score:  int   # 0-10
    recommendation: str

# ── Build the validation prompt ───────────────────────────────
def build_validation_prompt(sample_json: str, schema_description: str) -> str:
    return (
        'You are a senior data engineer reviewing a data sample before ingestion.\n\n'
        f'Schema description: {schema_description}\n\n'
        f'Data sample (JSON):\n{sample_json}\n\n'
        'Respond ONLY with a JSON object:\n'
        '{\n'
        '  "schema_match": true or false,\n'
        '  "anomalies": ["list", "of", "issues"],\n'
        '  "quality_score": 0-10,\n'
        '  "recommendation": "one sentence action"\n'
        '}'
    )


good_sample = [
    {'customer_id': 1, 'email': 'alice@co.com', 'age': 32, 'spend': 1200.0},
    {'customer_id': 2, 'email': 'bob@co.com',   'age': 28, 'spend':  850.5},
]
prompt_good = build_validation_prompt(
    json.dumps(good_sample, indent=2),
    'customer records: customer_id (int), email (str), age (int > 0), spend (float >= 0)'
)

print('=== Validation Prompt (first 400 chars) ===')
print(prompt_good[:400])
print('\n[...response would be a ValidationReport JSON object...]')

if GROQ_API_KEY:
    from langchain_groq import ChatGroq
    from langchain_core.messages import HumanMessage
    llm = ChatGroq(model='llama-3.1-70b-versatile', api_key=GROQ_API_KEY, temperature=0)
    raw = llm.invoke([HumanMessage(content=prompt_good)]).content
    start = raw.find('{'); end = raw.rfind('}') + 1
    report = ValidationReport(**json.loads(raw[start:end]))
    print(f'\nLive report: score={report.quality_score}, anomalies={report.anomalies}')
else:
    print('\n(GROQ_API_KEY not set -- live call skipped)')

# ══════════════════════════════════════════════════════════════
#  EXERCISE 7
# ══════════════════════════════════════════════════════════════
#
#  A raw employee dataset has intentional quality issues.
#  Write validate_employee_data(df) -> ValidationReport that:
#
#    1. Takes a 5-row sample of the DataFrame as JSON
#    2. Builds a prompt asking the LLM to check against the schema:
#         employee_id (int), name (str), age (int > 18 < 100),
#         salary (float > 0), email (str, must contain @)
#    3. If GROQ_API_KEY is set: call Groq and parse the response
#       into a ValidationReport Pydantic model
#    4. If not set: return a mock report with the known issues
#    5. Print: quality_score, number of anomalies, recommendation
#
#  Expected output (actual LLM will vary):
#    Quality score : 3/10
#    Anomalies     : 3
#      - age -5 is invalid (negative)
#      - salary 0 is invalid
#      - email 'not-an-email' missing @
#    Recommendation: Reject this batch and fix source data before re-ingesting.
#
# --- starting data ---

In [ ]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel

load_dotenv()
GROQ_API_KEY = os.environ.get('GROQ_API_KEY', '')

class ValidationReport(BaseModel):
    schema_match:   bool
    anomalies:      list
    quality_score:  int
    recommendation: str

# Raw data with intentional quality issues
raw_employees = pd.DataFrame({
    'employee_id': [1, 2, 3, 4, 5],
    'name':        ['Alice', 'Bob', None, 'Dave', 'Eve'],
    'age':         [32, -5, 28, 150, 40],
    'salary':      [95000, 0, 72000, 85000, -500],
    'email':       ['alice@co.com', 'bob@co.com', 'carol@co.com', 'not-an-email', 'eve@co.com'],
})

# ══════════════════════════════════════════════════════════════
#  CONCEPT 8 — PYDANTICAI: TYPED AGENT FRAMEWORK
# ══════════════════════════════════════════════════════════════
#
#  Concept 4 showed manual Pydantic parsing: ask for JSON in the
#  prompt, call json.loads(), then validate with Pydantic. It works,
#  but you manage JSON extraction and retry logic yourself.
#
#  PydanticAI (from the Pydantic team) wraps the LLM + validation:
#
#    agent = Agent('groq:llama-3.1-70b-versatile', result_type=MyModel)
#    result = agent.run_sync('some input')
#    record = result.data   # already a MyModel instance — no json.loads
#
#  If the LLM returns invalid JSON or missing fields, PydanticAI
#  retries the call automatically before raising an error.
#
#  WHEN TO USE PYDANTICAI VS LANGCHAIN:
#    PydanticAI — tight type contracts, minimal boilerplate, Pydantic-native
#                 best for: extraction, classification, validated structured output
#    LangChain  — RAG, complex multi-step chains, many integrations, LangSmith
#                 best for: retrieval pipelines, complex agents, LangSmith tracing
#
#  AGENT ANATOMY:
#    Agent(model, result_type, system_prompt, deps_type)
#      model        — 'openai:gpt-4o-mini', 'groq:llama-3.1-70b-versatile'
#      result_type  — Pydantic model: LLM MUST return a valid instance
#      system_prompt — persona and rules
#      deps_type    — inject runtime dependencies (DB conn, API client)
#
#  INSTALL: pip install pydantic-ai
#
#  EXAMPLE ----------------------------------------------------------

In [ ]:
print()
print("=" * 55)
print("CONCEPT 8: PydanticAI — Typed Agent Framework")
print("=" * 55)

from pydantic import BaseModel

# Step 1: Define the result schema — what the agent MUST return
class IncidentReport(BaseModel):
    severity:    str    # "low", "medium", "high", "critical"
    category:    str    # "network", "database", "application", "security"
    summary:     str
    action:      str
    needs_human: bool

# Step 2: Declare the agent (structure only — live call needs pydantic-ai + API key)
# from pydantic_ai import Agent
# agent = Agent(
#     'groq:llama-3.1-70b-versatile',
#     result_type=IncidentReport,
#     system_prompt=(
#         'You are an incident triage specialist. '
#         'Classify each incident and recommend an immediate action.'
#     )
# )
# result = agent.run_sync('Database connection pool exhausted — 50 users affected')
# report = result.data   # type: IncidentReport — IDE autocomplete works here

# Without pydantic-ai installed — simulate what agent.run_sync().data returns:
mock_response = {
    "severity":    "high",
    "category":    "database",
    "summary":     "Connection pool exhausted, blocking 50 users",
    "action":      "Restart pool, investigate connection leak in query layer",
    "needs_human": True,
}
report = IncidentReport(**mock_response)
print(f"Severity:    {report.severity}")
print(f"Category:    {report.category}")
print(f"Summary:     {report.summary}")
print(f"Action:      {report.action}")
print(f"Human:       {report.needs_human}")
print()
print("Manual approach (Concept 4) vs PydanticAI:")
print("  Manual:     call_llm() -> json.loads() -> Pydantic(**dict)")
print("              breaks if LLM wraps JSON in text or omits a field")
print("              no automatic retry on validation failure")
print("  PydanticAI: agent.run_sync() -> result.data  (typed instance)")
print("              retries automatically if LLM output fails validation")

# ══════════════════════════════════════════════════════════════
#  EXERCISE 8
# ══════════════════════════════════════════════════════════════
#
#  A BI team needs a SalesInsightAgent that reads quarterly data
#  and returns a validated SalesInsight report.
#
#  1. Define SalesInsight(BaseModel) with these 5 fields:
#       top_product       : str
#       revenue_total     : int
#       trend             : str   ("up", "down", "flat")
#       risk_flag         : str
#       recommended_action: str
#
#  2. Write the agent declaration as comments (structure only):
#       # agent = Agent(
#       #     'groq:llama-3.1-70b-versatile',
#       #     result_type=SalesInsight,
#       #     system_prompt='You are a BI analyst...',
#       # )
#
#  3. Parse mock_llm_output as a SalesInsight and print each field
#     with a label. Format revenue_total with comma separators.
#
#  4. Try to parse broken_output (missing the "trend" field).
#     Wrap in try/except ValidationError and print:
#       "ValidationError caught: 'trend' field is required"
#       "PydanticAI would retry the LLM call automatically."
#
#  Expected output:
#      Top product:   Laptop Pro
#      Revenue total: 184,500 NIS
#      Trend:         up
#      Risk flag:     High return rate in Electronics (12%)
#      Action:        Investigate logistics partner in Electronics
#      ---
#      ValidationError caught: 'trend' field is required
#      PydanticAI would retry the LLM call automatically.
#
# --- starting data ---

In [ ]:
from pydantic import BaseModel, ValidationError

# Mock response — simulates what agent.run_sync('Analyze Q3 sales').data returns
mock_llm_output = {
    "top_product":        "Laptop Pro",
    "revenue_total":      184500,
    "trend":              "up",
    "risk_flag":          "High return rate in Electronics (12%)",
    "recommended_action": "Investigate logistics partner in Electronics",
}

# Broken response — the LLM omitted the "trend" field
broken_output = {
    "top_product":        "Laptop Pro",
    "revenue_total":      184500,
    "risk_flag":          "High return rate in Electronics (12%)",
    "recommended_action": "Investigate logistics partner in Electronics",
}